<a href="https://colab.research.google.com/github/AngelsandDevsLOL/MelodiClass/blob/main/MelodiClass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math, random
import torch
import torchaudio
from torchaudio import transforms
from IPython.display import Audio

In [ ]:
from google.colab import files
uploaded = files.upload() # Kaggle API thing that you need to download from Kaggle

!ls -lha kaggle.json
!pip install -q kaggle # installing the kaggle package
!mkdir -p ~/.kaggle # creating .kaggle folder where the key should be placed
!cp kaggle.json ~/.kaggle/ # move the key to the folder
!pwd # checking the present working directory

!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d abdulvahap/music-instrunment-sounds-for-classification -p /content/data
!unzip /content/data/music-instrunment-sounds-for-classification.zip -d /content

!rm -rf /content/music_dataset/Banjo
!rm -rf /content/music_dataset/Dobro
!rm -rf /content/music_dataset/Floor_Tom
!rm -rf /content/music_dataset/Harmonica
!rm -rf /content/music_dataset/Hi_Hats
!rm -rf /content/music_dataset/Keyboard
!rm -rf /content/music_dataset/Mandolin
!rm -rf /content/music_dataset/Shakers
!rm -rf /content/music_dataset/cowbell
!rm -rf /content/music_dataset/vibraphone
!rm -rf /content/data
!rm -rf /content/sample_data

Saving kaggle.json to kaggle (1).json
-rw-r--r-- 1 root root 72 Mar 15 19:53 kaggle.json
/content
Dataset URL: https://www.kaggle.com/datasets/abdulvahap/music-instrunment-sounds-for-classification
License(s): apache-2.0
 99% 4.56G/4.59G [01:44<00:02, 18.8MB/s]
100% 4.59G/4.59G [01:44<00:00, 47.3MB/s]
Archive:  /content/data/music-instrunment-sounds-for-classification.zip
replace /content/music_dataset/Accordion/100.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
class_id_mapping = {'Accordion': 0, 'Acoustic_Guitar': 1, 'Bass_Guitar': 2, 'Clarinet': 3, 'Cymbals': 4, 'Drum_set': 5, 'Electro_Guitar': 6, 'flute': 7, 'Harmonium': 8, 'Horn': 9, 'Organ': 10, 'Piano': 11, 'Saxophone': 12, 'Tambourine': 13, 'Trombone': 14, 'Trumpet': 15, 'Ukulele': 16, 'Violin': 17}

from pathlib import Path

data_path = Path.cwd()/'music_dataset'

file_paths_list = []
class_ids_list = []

if not data_path.exists():
    print(f"Warning: The directory '{data_path}' does not exist. Please ensure it's created and contains data.")
else:
    for file_path in data_path.rglob('*'):
        if file_path.is_file():
            subfolder_name = file_path.parent.name

            if subfolder_name in class_id_mapping:
                class_id = class_id_mapping[subfolder_name]
                relative_path = "/" + str(file_path.relative_to(data_path))

                file_paths_list.append(relative_path)
                class_ids_list.append(class_id)
            else:
                print(f"Warning: Subfolder '{subfolder_name}' for file '{file_path}' not found in class_id_mapping. Skipping file.")

print(f"Collected {len(file_paths_list)} file paths and {len(class_ids_list)} class IDs.")

Collected 30862 file paths and 30862 class IDs.


In [ ]:
import pandas as pd

metadata_df = pd.DataFrame({
    'file_path': file_paths_list,
    'class_id': class_ids_list
})

print("Metadata DataFrame created successfully.")
print(metadata_df.head())

Metadata DataFrame created successfully.
         file_path  class_id
0   /Organ/186.wav        10
1   /Organ/317.wav        10
2  /Organ/1069.wav        10
3   /Organ/162.wav        10
4  /Organ/1288.wav        10


In [ ]:
metadata_df.to_csv('metadata.csv', index=False)
print("Metadata DataFrame saved to 'metadata.csv' successfully.")

Metadata DataFrame saved to 'metadata.csv' successfully.


In [ ]:
class AudioUtil():
  # Load an audio file and return the signal as a tensor and the sample rate (sr)
  @staticmethod
  def open(audio_file):
    sig, sr = torchaudio.load(audio_file)
    return (sig, sr)

  @staticmethod
  def rechannel (aud, new_channel):
    sig, sr = aud
    if (sig.shape[0] == new_channel):
      # the # of channels already matches the number of channels we want
      return aud # do nothing basically

    if (new_channel == 1):
      # convert from stereo to mono by only selecting the first channel
      resig = sig[:1, :]
    else:
      # convert from mono to stereo by duplicating it using the pytorch.concatenate:
      resig = torch.cat([sig, sig])
    return ((resig, sr))

  @staticmethod
  def resample(aud, newsr):
    # checks to make sure all files are sampled at the same rate
    sig, sr = aud

    if (sr == newsr):
      # sample rate already equals the desired rate
      return aud

    # otherwise (resamples the channels one-by-one)
    num_channels = sig.shape[0]
    resig = torchaudio.transforms.Resample(sr, newsr)(sig[:1,:])
    if (num_channels > 1):
      retwo = torchaudio.transforms.Resample(sr, newsr)(sig[1:,:])
      resig = torch.cat([resig, retwo])

    return ((resig, newsr))

  @staticmethod
  def time_shift(aud, shift_limit):
    # shifts the audio elements by a random amount
    sig, sr = aud
    _, sig_len = sig.shape
    shift_amt = int(random.random() * shift_limit * sig_len)
    return (sig.roll(shift_amt), sr)

  @staticmethod
  def spectro_gram(aud, n_mels=64, n_fft=1024, hop_len=None):
    #generate a spectrogram
    sig, sr = aud
    top_db = 80

    #spec is a tensor with shape [channel, n_mels, time]
    spec = transforms.MelSpectrogram(sr, n_fft=n_fft, hop_length=hop_len, n_mels=n_mels)(sig)

    #convert to decibels
    spec = transforms.AmplitudeToDB(top_db=top_db)(spec)
    return (spec)

  @staticmethod
  def spectro_augment(spec, max_mask_pct=0.1, n_freq_masks=1, n_time_masks=1):
    #augment the spectrogram by masking out sections of it to be replaced with the mean value
    _, n_mels, n_steps = spec.shape
    mask_value = spec.mean()
    aug_spec = spec

    freq_mask_param = max_mask_pct * n_mels
    for _ in range(n_freq_masks):
      aug_spec = transforms.FrequencyMasking(freq_mask_param)(aug_spec, mask_value)

    time_mask_param = max_mask_pct * n_steps
    for _ in range(n_time_masks):
      aug_spec = transforms.TimeMasking(time_mask_param)(aug_spec, mask_value)

    return aug_spec

In [ ]:
from torch.utils.data import DataLoader, Dataset, random_split
import torchaudio

# Data set

class SoundDS(Dataset):
  def __init__(self, df, data_path):
    self.df = df
    self.data_path = str(data_path)
    self.sr = 44100
    self.channel = 2
    self.shift_pct = 0.4

  def __len__(self): # number of items in the dataset
    return len(self.df)

  def __getitem__(self, index): # get ith item in the dataset
    audio_file = self.data_path + self.df.loc[index, 'file_path']
    class_id = self.df.loc[index, 'class_id']

    aud = AudioUtil.open(audio_file)
    reaud = AudioUtil.resample(aud, self.sr)
    rechan = AudioUtil.rechannel(reaud, self.channel)

    shift_aud = AudioUtil.time_shift(rechan, self.shift_pct)
    sgram = AudioUtil.spectro_gram(shift_aud, n_mels = 64, n_fft = 1024, hop_len = None)
    aug_sgram = AudioUtil.spectro_augment(sgram, max_mask_pct = 0.1, n_freq_masks = 2, n_time_masks = 2)

    return aug_sgram, class_id

In [ ]:
from torch.utils.data import random_split

# my_dataset = Dataset(all the data, data path)
myds = SoundDS(metadata_df, data_path)

# how many entries in our dataset
num_items = len(myds)
# 80% of those entries = how large is our training set
num_train = round(num_items *0.8)
# 20% of those entries = how large is our validation set
num_val = num_items - num_train
# splitting the dataset into the training and validation one
train_ds, val_ds = random_split(myds, [num_train, num_val])

# Create training and validation data loaders
train_dl = torch.utils.data.DataLoader(train_ds, batch_size = 16, shuffle = True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False)

In [ ]:
import torch.nn.functional as F
from torch.nn import init
import torch.nn as nn
import torch

#Audio classification model

class AudioClassifier(nn.Module):
  #Build model architecture

  def __init__(self):
    super().__init__()
    conv_layers = []

    #First convolution block with Relu and batch norm, using Kaiming Intiatization
    self.conv1 = nn.Conv2d(2, 8, kernel_size=(5,5), stride=(2,2), padding=(2,2))
    self.relu1 = nn.ReLU()
    self.bn1 = nn.BatchNorm2d(8)
    init.kaiming_normal_(self.conv1.weight, a=0.1)
    self.conv1.bias.data.zero_()
    conv_layers += [self.conv1, self.relu1, self.bn1]

    #Second convolution block
    self.conv2 = nn.Conv2d(8, 16, kernel_size=(3,3), stride=(2,2), padding=(1,1))
    self.relu2 = nn.ReLU()
    self.bn2 = nn.BatchNorm2d(16)
    init.kaiming_normal_(self.conv2.weight, a=0.1)
    self.conv2.bias.data.zero_()
    conv_layers += [self.conv2, self.relu2, self.bn2]

    #Third convolution block
    self.conv3 = nn.Conv2d(16, 32, kernel_size=(3,3), stride=(2,2), padding=(1,1))
    self.relu3 = nn.ReLU()
    self.bn3 = nn.BatchNorm2d(32)
    init.kaiming_normal_(self.conv3.weight, a=0.1)
    self.conv3.bias.data.zero_()
    conv_layers += [self.conv3, self.relu3, self.bn3]

    #Fourth convolution block
    self.conv4 = nn.Conv2d(32, 64, kernel_size=(3,3), stride=(2,2), padding=(1,1))
    self.relu4 = nn.ReLU()
    self.bn4 = nn.BatchNorm2d(64)
    init.kaiming_normal_(self.conv4.weight, a=0.1)
    self.conv4.bias.data.zero_()
    conv_layers += [self.conv4, self.relu4, self.bn4]

    #Linear classifier
    self.ap = nn.AdaptiveAvgPool2d(output_size=1)
    self.lin = nn.Linear(in_features=64, out_features=18)

    #Wrap the convlutional blocks
    self.conv = nn.Sequential(*conv_layers)

  # Forward pass computations
  def forward(self, x):
    #Run conv blocks
    x = self.conv(x)

    #Adaptive pool and flatter for input to linear layer
    x = self.ap(x)
    x = torch.flatten(x, 1)

    #Linear layer
    x = self.lin(x)

    return x

#Create the model and put it on the GPU if available
myModel = AudioClassifier()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
myModel = myModel.to(device)

#Check that it is on Cuda
next(myModel.parameters()).device

device(type='cuda', index=0)

In [ ]:
# Training loop ?
def training (model, train_dl, num_epochs):
  # The "Loss" Function - how 'wrong' are the model's guesses?
  criterion = nn.CrossEntropyLoss()
  # lr = learning rate, Adam is used as the Optimizer that updates the model weights to reduce the loss
  # We use Adam b/c we have a classification problem
  optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
  # We use OneCycleLR as the scheduler - manages the LR so that we have a max LR, and it fluctuates
  scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr = 0.001,
                                                  steps_per_epoch = int(len(train_dl)), epochs = num_epochs, anneal_strategy = 'linear')
  # Repeat for each epoch
  for epoch in range(num_epochs):
    running_loss = 0.0
    correct_prediction = 0
    total_prediction = 0

    # for each batch of data in the training set, i is the index, data is a tuple of values in the batch, and train_dl is the dataloader
    for i, data in enumerate(train_dl):
      # split data tuple into 2 tensor objects: moving it from the current device (CPU?) to the GPU
      # we want all tensors to be on the same device ..?
      inputs, labels = data[0].to(device), data[1].to(device)

      # .mean() finds average, .std() is standard deviation
      inputs_m, inputs_s = inputs.mean(), inputs.std()
      # "Normalization" = so that columns with large ranges don't dominate our model
      # Z-score standardization
      inputs = (inputs - inputs_m) / inputs_s

      # "Zero the parameter gradients" b/c we want to prevent double-counting over training iterations
      # must zero before we compute NEW gradients
      optimizer.zero_grad()

      # get the outputs
      outputs = model(inputs)
      # calculate our losses
      loss = criterion(outputs, labels)
      # (back propagation)
      # adds the new gradient to any existing gradients and calculates the derivative (gradient) of the loss for each parameter
      loss.backward()
      # iterates over all parameters + updates them using the optimization algorithm (in this case, Adam)
      # updates the calculated gradients
      optimizer.step()
      # updates the learning rate of the optimizer
      scheduler.step()

      # accumulate our losses
      running_loss += loss.item()

      # makes our prediction of the predicted class w/ the highest score
      # stores a tensor of the predicted class indices for the batch
      _, prediction = torch.max(outputs, 1)
      # Count of predictions that match the target label
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]

      # every 10 mini-batches,
      if i % 10 == 0:
        print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / 10))

    num_batches = len(train_dl)
    avg_loss = running_loss / num_batches
    # acc = accuracy
    acc = correct_prediction / total_prediction
    print(f'Epoch: {epoch}, Loss:{avg_loss:.2f}, Accuracy: {acc:.2f}')
  print('Finished Training')

num_epochs = 2 # we need to inc this !!!!
training(myModel, train_dl, num_epochs)

[1,     1] loss: 0.252
[1,    11] loss: 2.852
[1,    21] loss: 5.468
[1,    31] loss: 8.116
[1,    41] loss: 10.643
[1,    51] loss: 13.196
[1,    61] loss: 15.736
[1,    71] loss: 18.200
[1,    81] loss: 20.679
[1,    91] loss: 23.132
[1,   101] loss: 25.595
[1,   111] loss: 28.045
[1,   121] loss: 30.445
[1,   131] loss: 32.829
[1,   141] loss: 35.173
[1,   151] loss: 37.305
[1,   161] loss: 39.470
[1,   171] loss: 41.558
[1,   181] loss: 43.609
[1,   191] loss: 45.721
[1,   201] loss: 47.614
[1,   211] loss: 49.536
[1,   221] loss: 51.363
[1,   231] loss: 53.221
[1,   241] loss: 55.103
[1,   251] loss: 56.839
[1,   261] loss: 58.603
[1,   271] loss: 60.317
[1,   281] loss: 61.894
[1,   291] loss: 63.483
[1,   301] loss: 65.087
[1,   311] loss: 66.567
[1,   321] loss: 68.001
[1,   331] loss: 69.437
[1,   341] loss: 70.887
[1,   351] loss: 72.297
[1,   361] loss: 73.642
[1,   371] loss: 75.044
[1,   381] loss: 76.432
[1,   391] loss: 77.858
[1,   401] loss: 79.162
[1,   411] loss: 80.

In [ ]:
# Inference
def inference (model, val_dl):
  correct_prediction = 0
  total_prediction = 0

  # Disable gradient updates
  with torch.no_grad():
    for data in val_dl:
      # Get the input features and target labels, put them on the GPU
      inputs, labels = data[0].to(device), data[1].to(device)

      # Normalize the inputs
      inputs_m, inputs_s = inputs.mean(), inputs.std()
      inputs = (inputs - inputs_m) / inputs_s

      # Get predictions
      outputs = model(inputs)

      # Get the predicted clas with the highest score
      _, prediction = torch.max(outputs, 1)
      # Count of predictions that matched the target labl
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]

    acc = correct_prediction/total_prediction
    print(f'Accuracy: {acc:.2f}, Total items = {total_prediction}')

# Run inference on trained model with the validation set
inference(myModel, val_dl)

Accuracy: 0.89, Total items = 6172
